<a href="https://colab.research.google.com/github/krgyaan/LLM-Finetuning/blob/main/Instruction_finetuning_on_domain_specific_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- Pretrain Model(LLAM)
- Non Instrunction finetuning on plain text
- Instruction finetuning on instrunction data
- then next Preference Alignment



In [1]:
!pip install -U transformers peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.7 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
!pip install -U trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 10.4 MB/s eta 0:00:00


In [3]:
!pip install -U PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 51.6 MB/s eta 0:00:00


In [44]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, TaskType, get_peft_model
from datasets import load_dataset

In [5]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [7]:
import zipfile
import os

zip_path = "/content/llm-gem-data.zip"

with zipfile .ZipFile(zip_path, 'r') as zip_ref:
  zip_ref.extractall()

In [39]:
trained_model_path = "/content/checkpoint-4/"

In [9]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")
nif_trained_model = PeftModel.from_pretrained(base_model, trained_model_path)

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

In [10]:
non_instruction_model = AutoModelForCausalLM.from_pretrained(trained_model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [11]:
prompt="Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"

In [14]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

inputs = tokenizer(prompt, return_tensors="pt").to(device)
non_instruction_model.to(device)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=2048, out_features=8, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=8, out_features=2048, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): lora.Linear(
            (base_layer): Linear(in_features=2048, out_features=256, bia

In [15]:
outputs = non_instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

print("\nModel Output: \n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Model Output: 

Clinical trials demonstrated that combining Atorvastatin with Ezetimibe, a drug that inhibits the absorption of cholesterol and helps reduce LDL-C levels, reduces LDL-C by 36% over six months compared to Atorvastatin alone.
The trial also showed that combining Atorvastatin with Ezetimibe results in an additional decrease of 21%. This combination therapy significantly decreases both total LDL-C and non-HDL-C at six months, as well as


### Now let's start Instruction Finetuning
- Start with inspecting built in data (Amod/mental_health_counseling_conversations)

In [16]:
from datasets import Dataset, load_dataset
dataset = load_dataset("Amod/mental_health_counseling_conversations", split="train")

README.md: 0.00B [00:00, ?B/s]

combined_dataset.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

In [25]:
dataset

Dataset({
    features: ['Context', 'Response'],
    num_rows: 3512
})

In [17]:
def format_row(data):
  que = data["Context"]
  ans = data["Response"]
  data["Text"] = f"[INST] {que} [/INST] {ans}"
  return data

In [30]:
formatted_data = dataset.map(format_row)

Map:   0%|          | 0/3512 [00:00<?, ? examples/s]

In [35]:
formatted_data[0]["Text"]

"[INST] I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone? [/INST] If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow terrible.B

In [36]:
import pandas as pd

# Convert dataset to dataframe
df = pd.DataFrame(dataset)

In [44]:
# You can convert it into any format now:
# df.to_csv("mental_health_counseling_conversations.csv", index=False)
df.to_json("mental_health_counseling_conversations.json", orient="records", lines=True)

In [46]:
# And after conversion you can re-load the data like:
from datasets import load_dataset

# dataset_from_csv = load_dataset("csv", data_files="/content/mental_health_counseling_conversations.csv", split="train")
dataset_from_json = load_dataset("json", data_files="/content/mental_health_counseling_conversations.json", split="train")

#### Now let's load our dataset

In [18]:
from datasets import load_dataset

own_dataset = load_dataset("csv", data_files="/content/pharma_instruction_data.csv", split="train")

Generating train split: 0 examples [00:00, ? examples/s]

In [19]:
def format_own_data(data):
  ins = data["instruction"]
  inp = data["input"]
  oup = data["output"]
  prompt = f"###Instruction:\n{ins}\n###Input:\n{inp}\n###Output:\n{oup}"
  return {"text": prompt}

In [20]:
formatted_own_data = own_dataset.map(format_own_data)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [22]:
formatted_own_data['text'][0]

'###Instruction:\nExplain the mechanism of action of Metformin.\n###Input:\nNone\n###Output:\nMetformin activates AMP-activated protein kinase (AMPK), which increases glucose uptake and fatty-acid oxidation while inhibiting hepatic gluconeogenesis, thereby lowering blood glucose.'

In [24]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [45]:
def tokenize_fn(example):
    tokens = tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [46]:
tokenized = formatted_own_data.map(tokenize_fn, batched=True)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [47]:
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

non_instruction_model.config.use_cache = False          # Required for gradient checkpointing
non_instruction_model.enable_input_require_grads()      # Required for LoRA + quantized models

In [48]:
from peft import prepare_model_for_kbit_training, get_peft_model, LoraConfig, TaskType

model = prepare_model_for_kbit_training(non_instruction_model)   # Must be called before get_peft_model

In [52]:
# Loaded Quantized Model

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map={"": 0}        # Force everything onto GPU 0, not "auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [53]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

In [55]:
qlora_model = get_peft_model(non_instruction_model, lora_config)
qlora_model.print_trainable_parameters()   # sanity check

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023


In [70]:
training_args = TrainingArguments(
    output_dir = "./llm-gem-data",    # Save model/checkpoints -> folder path -> where your model ends up
    num_train_epochs = 2,             # Training cycle -> 1-5 -> More = longer training
    per_device_train_batch_size = 2,  # Batch per GPU -> 1-8(Colab) -> Affects speed & memore
    save_steps = 500,                 # Save frequency -> 100-1000 -> More = frequent checkpoints
    save_total_limit = 2,             # Keep last N ->1-3 -> Saves disks
    logging_steps = 50,               # Log every X step -. 20-100  -> for progress monitoring
    learning_rate = 2e-5,             # Weight updates rate -> 1e-5 - 5e-5 ->Controls stability
    fp16 = True,                      # Mixed precision -> True -> Faster + less memory
    report_to = "none",               # Logging destination -> "none" -> Keeps it simple
    optim="paged_adamw_8bit",         # Memory-efficient optimizer for QLoRA
    gradient_checkpointing=True,
)

In [71]:
trainer = Trainer(
    model=qlora_model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=data_collator
)

In [72]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=6, training_loss=2.0422511100769043, metrics={'train_runtime': 3.9683, 'train_samples_per_second': 2.52, 'train_steps_per_second': 1.512, 'total_flos': 31814823444480.0, 'train_loss': 2.0422511100769043, 'epoch': 2.0})

In [ ]:
model_path = "/content/llm-gem-data/checkpoint-6/"

In [82]:
# Clear the cache and Garbage Collection

import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [85]:
from peft import PeftModel

instruction_model = PeftModel.from_pretrained(non_instruction_model, model_path)

# model = PeftModel.from_pretrained(
#     base_model,
#     "/content/llm-gem-data/checkpoint-6/"
# )

In [86]:
instruction_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_feat

In [95]:
prompt = "List two advantages of combining Atorvastatin with Ezetimibe."

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [96]:
outputs = instruction_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [97]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

List two advantages of combining Atorvastatin with Ezetimibe.
A: One advantage is the reduction in cardiovascular events. Another advantage is that it's easier to take at night when you're sleeping.
B: Both have an anti-cholesterol effect, but Ezetimibe has a different mechanism than Atorvastatin. It lowers cholesterol by slowing down the liver.
Q: Explain how lipid (fat) levels are measured.
A: A
